# Composite Transformers (ColumnTransformer & TransformedTargetRegressor)

## 1. Yeh Kya Hai? (What is it?)
Real-world datasets kabhi bhi ek jaise (homogeneous) nahi hote. Ek hi dataset me tumhare paas numbers (Age, Salary), categories (Gender, City), aur text (Reviews, Emails) sab ek sath hota hai.

Machine Learning models sirf numbers samajhte hain, isliye humein data ko transform karna padta hai:
*   **Numbers** ko *Scale* karna padta hai (taaki bade numbers model ko dominate na karein).
*   **Categories** ko *Encode* karna padta hai (One-Hot Encoding).
*   **Text** ko *Vectorize* karna padta hai.

**ColumnTransformer** ek aisi factory assembly line hai jo yeh saare kaam ek sath, ek hi step me karti hai. Tum isko batate ho ki "Bhai, column 0 ko scale kar de, column 1 ko encode kar de", aur yeh sabko process karke aakhri me ek single matrix me jod (concatenate) deta hai.

**TransformedTargetRegressor** bhi iska hi ek bhai hai. Jaise `ColumnTransformer` input features ($X$) ko transform karta hai, waise hi `TransformedTargetRegressor` target variable ($y$) ko transform karta hai (jaise target ka log lena) training se pehle, aur prediction ke time wapas original form me convert kar deta hai.

---

## 2. Hum Ise Kyu Use Karte Hain? (Why use it?)
1.  **Code Clean aur Professional Banta Hai:** Iske bina tumhe har column ko alag-alag slice karke transform karna padega, aur phir unhe `numpy.hstack` se jodna padega. Yeh bahut messy aur error-prone hota hai.
2.  **Data Leakage Rota Hai:** Jab tum isko `Pipeline` ke andar use karte ho, toh Cross-Validation ke time training aur test data completely separate rehte hain, aur model production-ready banta hai.
3.  **Heterogeneous Data Handling:** Ek hi object me multiple preprocessing techniques combine ho jati hain.

---

## 3. Maths aur Logic Iske Piche (How it works & The Math)

**A. ColumnTransformer ka Logic:**
Maan lo tumhare paas 3 columns hain: `[Age, Gender, Salary]`.
*   Age par $F_1$ (StandardScaler) lagaya.
*   Gender par $F_2$ (OneHotEncoder) lagaya.
*   Salary par kuch nahi lagaya (pass through).

ColumnTransformer in sabka output nikalega aur unhe horizontally concatenate kar dega:
$$ Output = [ \ F_1(Age) \ \ | \ \ F_2(Gender) \ \ | \ \ Salary \ ] $$

Isme ek `remainder` parameter hota hai:
*   `remainder='drop'`: Jo columns list me nahi hain, unko hata do.
*   `remainder='passthrough'`: Jo columns list me nahi hain, unko bina chhede output matrix me waise hi jod do.

**B. TransformedTargetRegressor ki Maths:**
Maan lo tumhara target $y$ (Salary) bahut skewed hai (kisi ki salary 10k hai, kisi ki 10 Crore). Aise me linear models fail ho jate hain. Hum target ka Logarithm le lete hain taaki data normal ho jaye.
*   **Training Time:** Model $X$ aur $y'$ par train hota hai, jahan $$ y' = \log(y) $$
*   **Prediction Time:** Model pehle $\hat{y}'$ predict karta hai, phir usko original scale me laane ke liye inverse function ($e^x$) lagata hai: $$ \hat{y} = e^{\hat{y}'} $$

---

## 4.Implementation Code

Niche diya gaya code ekdam production-ready hai. Isme `ColumnTransformer` aur `TransformedTargetRegressor` dono ko ek `Pipeline` me use kiya gaya hai.

*(Note: Tumhare original notes me age par `CountVectorizer` likha tha, wo galat tha. CountVectorizer text ginta hai. Age ek number hai toh wahan `StandardScaler` aayega. Maine yahan correct kar diya hai.)*



In [5]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# 1. Dummy Dataset Banana (Heterogeneous Data)
data = {
    'age': [25, 32, 47, 51, 22],               # Numeric
    'gender': ['M', 'F', 'F', 'M', 'M'],       # Categorical
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Pune', 'Mumbai'], # Categorical
    'salary': [50000, 80000, 120000, 150000, 45000] # Target (y) - skewed
}
df = pd.DataFrame(data)

X = df[['age', 'gender', 'city']]
y = df['salary']

# 2. ColumnTransformer Setup
# Rule: Age scale hogi, Gender aur City One-Hot Encode honge
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), ['age']),
        ('cat_encoder', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
    ],
    remainder='drop', # Agar in 3 ke alawa koi column hota toh wo delete ho jata
    verbose_feature_names_out=False
)

# 3. TransformedTargetRegressor Setup
# Salary skewed hoti hai, isliye Model ko train karne se pehle y ka log(y) lenge
model_with_target_transform = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,      # Training ke time y = log(1+y) karega
    inverse_func=np.expm1 # Predict karne ke baad y = e^y - 1 karega (wapas original)
)

# 4. Final Pipeline Create Karna
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model_with_target_transform)
])

# 5. Model Train aur Predict Karna
# Dekho, kitna clean hai! Fit karte hi transformation aur log target sab apne aap ho gaya
pipeline.fit(X, y)

# Prediction karte time target automatically log se wapas original rupees me aayega
predictions = pipeline.predict(X)

print("Original Salaries: ", y.values)
print("Predicted Salaries: ", np.round(predictions, 2))

Original Salaries:  [ 50000  80000 120000 150000  45000]
Predicted Salaries:  [ 50000.  80000. 120000. 150000.  45000.]
